In [1]:
import pandas as pd
import numpy as np
import  matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [2]:
# import databasse/data

conn=sqlite3.connect('customer_churn.db')
sql_query =""" 
        SELECT name
        FROM sqlite_master
        WHERE type ='table' """
tables = pd.read_sql(sql_query,conn)

# create dataframe for each tables
for table_name in tables['name']:
    df=pd.read_sql(f"SELECT * FROM {table_name}",conn)
    globals()[f"df_{table_name}"] =df
    print(f"Created dataframe:df_{table_name}")
    
conn.close()

Created dataframe:df_db_customer
Created dataframe:df_db_subscription
Created dataframe:df_db_support


In [3]:
#print table names and column names

conn=sqlite3.connect(r'C:\Users\Mariya\OneDrive\Desktop\intership\HotstarAnalysis\customer_churn.db')
for table_name in tables['name']:
    print(f"\nTable Name:{table_name}")
    #get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns=pd.read_sql(columns_query,conn)
    print("Columns:")
    print(columns['name'].tolist())
conn.close()


Table Name:db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name:db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name:db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


data cleaning

In [5]:
#data cleaning
#df_db_customer.head()
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,None,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,None,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,None,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,None,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,None,None


In [6]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     object
 1   name        21 non-null     object
 2   country     18 non-null     object
 3   state       21 non-null     object
 4   gender      21 non-null     object
 5   dob         21 non-null     object
 6   interests   4 non-null      object
 7   pincode     0 non-null      object
dtypes: object(8)
memory usage: 1.4+ KB


# a.rename columns -name to Customer_name
# b.drop columns - interests and pincode
# c.change data type -dob
# d.data standardisation -gender
# e.fix missing value -country

In [7]:
# a.rename columns -name to Customer_name
df_db_customer.rename(columns ={'name':'customer_name'},inplace=True)
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [8]:
#b.drop columns - interests and pincode
df_db_customer.drop(columns=['interests','pincode'],axis=0,inplace=True)

In [9]:
# c.change data type -dob
df_db_customer['dob']=pd.to_datetime(df_db_customer['dob'])

In [10]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     object        
 1   customer_name  21 non-null     object        
 2   country        18 non-null     object        
 3   state          21 non-null     object        
 4   gender         21 non-null     object        
 5   dob            21 non-null     datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 1.1+ KB


In [11]:
# d.data standardisation -gender
#df_db_customer['gender'].unique()
df_db_customer['gender']=df_db_customer['gender'].replace({'Women':'Female','Men':'Male'})

In [12]:
df_db_customer['gender'].unique()

array(['Male', 'Female'], dtype=object)

In [13]:
# e.fix missing value -country
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,None,Delhi,Female,1988-12-10
8,0015-UOCOJ,maya,None,Kathmandu,Female,1985-07-07
12,0018-NYROU,chitra,None,Telangana,Female,2004-12-01


In [18]:
# counntry and state - unique value pair
state_country_mapping=df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

df_db_customer['country']=df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [22]:
df_db_customer['country']

0     India
1     India
2     India
3     India
4     India
5     India
6     India
7     India
8     Nepal
9     Nepal
10    India
11    India
12    India
13    India
14    India
15    India
16    India
17    India
18    India
19    India
20    India
Name: country, dtype: object

# db_subscription table data cleaning

In [23]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,None,None,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,None,None,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,None,None,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [24]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     object 
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     object 
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     object 
 5   contract_type            21 non-null     object 
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      object 
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.9+ KB


In [30]:
# change data type object to datetime
date_col=['subscription_start_date','cancellation_date','renewal_date']

df_db_subscription[date_col]=df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     object        
 1   subscription_start_date  21 non-null     datetime64[ns]
 2   subscription_type        21 non-null     object        
 3   renewal_date             21 non-null     datetime64[ns]
 4   plan_type                21 non-null     object        
 5   contract_type            21 non-null     object        
 6   cancellation_date        6 non-null      datetime64[ns]
 7   cancellation_reason      6 non-null      object        
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[ns](3), float64(1), int64(2), object(5)
memory usage: 1.9+ KB


#df_db_support table DATA Cleaning

In [31]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,None
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,None


In [32]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      object
 1   complaint_date  9 non-null      object
 2   escalations     9 non-null      object
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      object
dtypes: int64(1), object(5)
memory usage: 564.0+ bytes


In [33]:
df_db_support.drop(columns=['col_1','comment'],inplace=True)

In [34]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      object
 1   complaint_date  9 non-null      object
 2   escalations     9 non-null      object
 3   csat_score      9 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 420.0+ bytes


In [35]:
df_db_support['complaint_date']=pd.to_datetime(df_db_support['complaint_date'])
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      object        
 1   complaint_date  9 non-null      datetime64[ns]
 2   escalations     9 non-null      object        
 3   csat_score      9 non-null      int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 420.0+ bytes


#Feature Engineering and data Analysis

In [37]:
# create a new col using existing col - churn flag
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88,1


In [38]:
# first fix support table duplicates then merge
df = (df_db_subscription
            .merge(df_db_customer,on ='customerid' , how='left')
            .merge(df_db_support,on ='customerid' , how='left'))

In [42]:
df_db_subscription.shape

(21, 12)

In [43]:
df.shape # two recod extry 

(23, 20)

In [44]:
df_db_subscription['customerid'].nunique()

21

In [45]:
df_db_customer['customerid'].nunique()

21

In [46]:
df_db_support['customerid'].nunique()

7

In [47]:
df_db_support['customerid'].size    # in this reco extra

9

In [ ]:
#Grouping by custerid
df_db_support['complaint_count']=df_db_support.groupby('customerid')['customerid'].transform('count')

In [54]:
df_db_support

,customerid,complaint_date,escalations,csat_score,complaint_count
0,0003-MKNFE,2024-08-28,N,60,2
1,0003-MKNFE,2024-08-28,Y,10,2
2,0013-EXCHZ,2024-01-20,Y,20,1
3,0013-MHZWF,2025-03-18,N,90,1
4,0013-SMEOE,2024-11-01,N,30,1
5,0017-IUDMW,2024-04-10,Y,25,1
6,0019-EFAEP,2024-09-27,Y,30,1
7,0022-TCJCI,2024-09-13,Y,10,2
8,0022-TCJCI,2024-09-14,N,90,2


In [55]:
df_db_support=df_db_support.sort_values('complaint_date').drop_duplicates('customerid',keep='last')

In [57]:
df_db_support

,customerid,complaint_date,escalations,csat_score,complaint_count
2,0013-EXCHZ,2024-01-20,Y,20,1
5,0017-IUDMW,2024-04-10,Y,25,1
1,0003-MKNFE,2024-08-28,Y,10,2
8,0022-TCJCI,2024-09-14,N,90,2
6,0019-EFAEP,2024-09-27,Y,30,1
4,0013-SMEOE,2024-11-01,N,30,1
3,0013-MHZWF,2025-03-18,N,90,1


In [56]:
df_db_support['customerid'].size

7

In [58]:
#merge df
df = (df_db_subscription
            .merge(df_db_customer,on ='customerid' , how='left')
            .merge(df_db_support,on ='customerid' , how='left'))

In [59]:
df.shape # join is successful

(21, 21)

#Data ANALYSIS

In [60]:
df.to_csv('exported_churn_data.csv',index=False)

In [61]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='object')

In [63]:
# 1.churn Rate
churn_rate=df['churn_flag'].mean()*100
print("Churn Rate:",round(churn_rate,2),"%")

Churn Rate: 28.57 %


In [65]:
#2.Retention Rate
retention_rate=100-churn_rate
print("Retention Rate:",round(retention_rate,2),"%")

Retention Rate: 71.43 %


In [66]:
# 3. churn by plan type
churn_by_plan=df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_by_plan')
print(churn_by_plan)

  plan_type  churn_by_plan
0     Basic          60.00
1   Premium          14.29
2  Standard          22.22


In [97]:
# 4.churn by state  = sum(revenue) and count of user ----#####
churn_by_state = (
    df.groupby('state')
    .agg(
        total_users=('customerid', 'count'),
        churned_users=('churn_flag', 'sum'),
        total_revenue=('monthly_charges', 'sum')
    )
    .reset_index()
)

print(churn_by_state)

           state  total_users  churned_users  total_revenue
0          Delhi            4              1          52.96
1      Karnataka            2              2          20.98
2      Kathmandu            2              0          20.98
3    Maharashtra            3              0          50.97
4      Meghalaya            3              2          42.97
5       Nagaland            1              0          22.99
6      Rajasthan            2              0          36.98
7      Telangana            2              1          30.98
8  Uttar Pradesh            2              0         115.98


In [98]:
# 5. churn by subscription type +sum(revenue) and count of users----##
churn_by_subscription = (
    df.groupby('subscription_type')
    .agg(
        total_users=('customerid', 'count'),
        churned_users=('churn_flag', 'sum'),
        total_revenue=('monthly_charges', 'sum')
    )
    .reset_index()
)

print(churn_by_subscription)

  subscription_type  total_users  churned_users  total_revenue
0           Organic            9              0         145.91
1              Paid            6              1         174.94
2          Refferal            6              5          74.94


In [76]:
#6. ARPU -Avg revenue per user
arpu=df['monthly_charges'].mean()
print('Avg Revenue Per User=',round(arpu,2))

Avg Revenue Per User= 18.85


In [80]:
#7 Avg Customer Tenure
# count of days user has used our service : concellation data else current date
today = pd.Timestamp.today()
df['tenure_days']=np.where(
    df['cancellation_date'].notna(),
    (df['cancellation_date']-df['subscription_start_date']).dt.days,
    (today-df['subscription_start_date']).dt.days
)
avg_tenure=df['tenure_days'].mean()
print("Avg Tenure (Days)= ",round(avg_tenure),0)

Avg Tenure (Days)=  1539 0


In [81]:
# 8 Revenue at risk -revenue last from churned users
revenue_at_risk=df.loc[df['churn_flag']==1,'monthly_charges'].sum()
print("Revenue at Risk (Rs'K') =",revenue_at_risk)

Revenue at Risk (Rs'K') = 73.94


In [82]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count',
       'customer_age', 'tenure_days'],
      dtype='object')

In [84]:
# 9.Esclation Rate
escalations_rate= (df['escalations']=='Y').mean()*100
print("Escalations Rate= ",round(escalations_rate,2),"%")

Escalations Rate=  19.05 %


In [85]:
# 10.Avg Complaint per user
avg_complaint = df['complaint_count'].sum() /df['customerid'].nunique()
print("Avg Complaint per user = ",round(avg_complaint,2))

Avg Complaint per user =  0.43


In [87]:
#11. Correlation Esclation Vs Churn
df[[,'escalations','churn_flag']].dropna() # escalations str type  so Y=1 ,N=0 doing compaerizan

,customerid,escalations,churn_flag
1,0003-MKNFE,Y,1
4,0013-EXCHZ,Y,1
5,0013-MHZWF,N,0
6,0013-SMEOE,N,1
11,0017-IUDMW,Y,1
13,0019-EFAEP,Y,1
18,0022-TCJCI,N,1


In [89]:
df['escalations']=np.where(df['escalations']=='Y',1,0) # encoding string to int type

In [90]:
df.head(2)

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,country,state,gender,dob,complaint_date,escalations,csat_score,complaint_count,customer_age,tenure_days
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,...,India,Maharashtra,Male,1982-04-12,NaT,0,NaN,NaN,44,2009.0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,India,Karnataka,Male,1995-11-23,2024-08-28,1,10.0,2.0,30,1501.0


In [91]:
corr_df=df[['escalations','churn_flag']].dropna()
correlation=corr_df['escalations'].corr(df['churn_flag'])
print("Correlation between escalations vs churn is = ",round(correlation,2))

Correlation between escalations vs churn is =  0.77


In [93]:
# 12. churn risk - create a colum using existhing col

conditions= [(df['churn_score']<50),
              (df['churn_score']>=50) & (df['churn_score']<75),
              (df['churn_score']>=70)
            ]
choices =['low','med','high']
df['churn_risk']=np.select(conditions,choices,default='unkown')

In [95]:
df[['churn_risk','churn_score']].tail()

,churn_risk,churn_score
16,med,62
17,low,27
18,high,99
19,low,7
20,low,47


In [108]:
# 13. Churn by Contract Type
churn_by_contract = (
    df.groupby('contract_type')['churn_flag']
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name='churn_rate')
)
print(churn_by_contract)

  contract_type  churn_rate
0        Annual        8.33
1       Monthly       55.56 %


# Conclusion 
1. Churn Rate: 28.57%
   → Overall, 28.57% of customers have churned.

2. Retention Rate: 71.43%
   → 71.43% of customers are retained by the company.

3. Churn by Plan Type:
   → Basic Plan has the highest churn rate at 60.00%.
   → Standard Plan has a churn rate of 22.22%.
   → Premium Plan has the lowest churn rate at 14.29%.

4. ARPU (Average Revenue Per User): Rs. 18.85
   → On average, each customer generates Rs. 18.85 in revenue.

5. Revenue at Risk: Rs. 73.94K
   → Rs. 73.94K revenue is at risk due to customer churn.

6. Escalation Rate: 19.05%
   → 19.05% of customers had escalated complaints or issues.

7. Average Complaints Per User: 0.43
   → On average, each customer has 0.43 complaints.

8. Correlation Between Escalations and Churn: 0.77
   → There is a strong positive correlation between escalations and customer churn.

9. Churn by Contract Type:
   → Monthly Contract has the highest churn rate at 55.56%.
   → Annual Contract has a low churn rate of 8.33%.